# 🍎 K-Means Clustering with the Fruits Dataset

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_06/kmeans_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_06/kmeans_clustering.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 


### Learning objectives

- Understand K-Means with a fixed value of $k$.
- Interpret cluster assignments and centroids.
- Use elbow and silhouette analyses to select $k$.
- Retrain K-Means with the selected number of clusters.

### Documentation

- [```KMeans```](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)

---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

### Imports

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import silhouette_score

from MLTools import evaluate_kmeans_range, plot_clustering_metric 

### Part I. K-Means with a constant k

#### Load the dataset

In [ ]:
CSV_PATH = _ 

fruits = pd.read_csv(CSV_PATH)

print("Shape:", fruits.shape)
fruits.head()


#### Inspect the data

In [ ]:
fruits.info()

In [ ]:
# Compute statistics
_ 


In [ ]:
# compute missing values

fruits. _ 

#### Select numerical features

K-Means requires numerical variables. The descriptive fruit columns are retained only for interpretation.


In [ ]:
feature_columns = [ _ ]

X = fruits[feature_columns].copy()
X.head()


In [ ]:
X_train, X_test = train_test_split( _ , test_size = _ , random_state = 42)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

#### K-Means idea

K-Means repeatedly assigns observations to their nearest centroid and recomputes each centroid.

$$
\text{Inertia}
=
\sum_{j=1}^{k}
\sum_{\mathbf{x}_i\in C_j}
\left\|\mathbf{x}_i-\boldsymbol{\mu}_j\right\|^2
$$


#### Train K-Means

In [ ]:
INITIAL_K = 3

initial_model = KMeans( n_clusters=INITIAL_K, random_state=7)

initial_model.fit( X_train.drop(["fruit_label"] , axis=1))
initial_clusters = initial_model.predict( X_test.drop(["fruit_label"], axis=1))

#### Inspect cluster assignments

In [ ]:
fruits_initial = X_test.copy()
fruits_initial["cluster"] = initial_clusters

fruits_initial.head(10)


In [ ]:
fruits_initial["cluster"].value_counts().sort_index()


#### Compare clusters with fruit names

In [ ]:
pd.crosstab( fruits_initial["cluster"], fruits_initial["fruit_label"] )


#### Centroids in original units

In [ ]:
initial_centroids = initial_model.cluster_centers_

initial_centroid_df = pd.DataFrame( initial_centroids, columns=feature_columns[:-1])

initial_centroid_df.index.name = "cluster"
initial_centroid_df.round(3)

#### Visualize clusters

In [ ]:
h_index, v_index = _ , _

plt.figure(figsize=(9, 6))
plt.scatter(
    X_test.iloc[:, h_index],
    X_test.iloc[:, v_index],
    c=initial_clusters,
    edgecolors="black",
    alpha=0.8,
    cmap = 'Paired'
)
plt.xlabel(feature_columns[h_index])
plt.ylabel(feature_columns[v_index])
plt.title(f"K-Means Clusters with k={INITIAL_K}")
plt.show()

# print("Explained variance:", pca.explained_variance_ratio_.sum())


### Part II. Select the optimal k

#### Evaluate candidate values of k

In [ ]:
candidate_clusters = _ 

kmeans_analysis = evaluate_kmeans_range(
    X_train = _ ,
    X_test = _ ,
    clusters = candidate_clusters,
    metric = "euclidean",
    random_state = 42
)

kmeans_analysis

#### Plots using for evaluating performance

In [ ]:
_, axes = plt.subplots(1,3,figsize=(15,4))

plot_clustering_metric(
    results=kmeans_analysis,
    x="n_clusters",
    y="mean_nearest_centroid_distance",
    title="K-Means Elbow Analysis",
    x_label="Number of clusters, k",
    y_label="Mean nearest-centroid distance", 
    ax = axes[0]
)

plot_clustering_metric(
    results=kmeans_analysis,
    x="n_clusters",
    y="inertia_train",
    title="K-Means Inertia",
    x_label="Number of clusters, k",
    y_label="Training inertia", 
    ax = axes[1]
)

plot_clustering_metric(
    results=kmeans_analysis,
    x="n_clusters",
    y="silhouette",
    title="K-Means Silhouette Analysis",
    x_label="Number of clusters, k",
    y_label="Silhouette score", 
    ax = axes[2]
)


#### Select k using the highest silhouette score

In [ ]:
best_row = kmeans_analysis.loc[ kmeans_analysis["silhouette"].idxmax() ]

optimal_k = int(best_row["n_clusters"])

print("Recommended k:", optimal_k)
print("Silhouette:", round(best_row["silhouette"], 3))
print("Mean distance:", round(best_row["mean_nearest_centroid_distance"], 3))
print("Inertia:", round(best_row["inertia_train"], 3))


#### Retrain K-Means with the selected k

The final model is fitted on the complete standardized dataset.


In [ ]:
final_model = KMeans( n_clusters = optimal_k, random_state = 42)

final_model.fit( _ )
final_clusters = final_model.predict( _ )

print("Final inertia:", final_model.inertia_)
print("Final silhouette:", silhouette_score( _ , _ ))


#### Inspect final assignments

In [ ]:
fruits_final = X_test.copy()
fruits_final["cluster"] = final_clusters

fruits_final.head(10)


In [ ]:
fruits_final["cluster"].value_counts().sort_index()

In [ ]:
pd.crosstab( fruits_final["cluster"], fruits_final["fruit_label"] )

#### Final centroids

In [ ]:
final_centroids = final_model.cluster_centers_

final_centroid_df = pd.DataFrame( final_centroids, columns=feature_columns[: -1] )

final_centroid_df.index.name = "cluster"
final_centroid_df.round(3)

#### Final visualization

In [ ]:
h_index, v_index = _ , _ 

plt.figure(figsize=(9, 6))
plt.scatter(
    X_test.iloc[:, h_index],
    X_test.iloc[:, v_index],
    c = final_clusters,
    edgecolors = "black",
    alpha = 0.8,
    cmap = 'Paired'
)
plt.xlabel(feature_columns[h_index])
plt.ylabel(feature_columns[v_index])
plt.title(f"Final K-Means Clusters with k={optimal_k}")
plt.show()


#### Optional export

In [ ]:
OUTPUT_PATH = "fruits_with_kmeans_clusters.csv"

fruits_final.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)


### Practice exercises

1. Repeat using only width, height, and color score.
2. Change `random_state`.
3. Describe the dominant fruit type and average characteristics in every cluster.


In [ ]:
# Your code here


---


## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *KMeans clustering: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.